# Cross-Species Knowledge Transfer: Analysis of Two Related Papers
## Relevance to the nAChR Variant Effect Predictor (GOF/LOF) Project

*Date: 2026-07-27*

**Papers reviewed:**
1. **GenePlexusZoo** — Mancuso CA, Johnson KA, Liu R, Krishnan A. "Joint representation of molecular networks from multiple species improves gene classification." *PLOS Computational Biology* (2024), 20(1):e1011773. DOI: [10.1371/journal.pcbi.1011773](https://doi.org/10.1371/journal.pcbi.1011773)
2. **Cross-species review** — Yuan H, Mancuso CA, Johnson K, Braasch I, Krishnan A. "Computational strategies for cross-species knowledge transfer." *Nature Methods* (2026), 23(2):312–327. DOI: [10.1038/s41592-025-02931-9](https://doi.org/10.1038/s41592-025-02931-9)

**Why your advisor recommended these:** Both papers are from the Krishnan lab and form a pair — GenePlexusZoo is the method paper, and the Nature Methods piece is the comprehensive review that puts it in context. Together they address the exact problem your project faces: **how to rigorously transfer knowledge from model organisms (mouse) to human for variant effect prediction.**

## 1. Paper 1: GenePlexusZoo — Multi-Species Network Embedding for Gene Classification

### What it does
GenePlexusZoo builds a **joint molecular network** spanning 6 species (human, mouse, zebrafish, fly, worm, yeast) by connecting genes across species using **eggNOG orthology groups**. It then runs **node2vec** (via PecanPy) on this combined network to produce a single, reusable low-dimensional embedding where each gene — regardless of species — lives in the same vector space.

### Key technical details
- **Networks used:** BioGRID (binary physical PPIs) + IMP (weighted functional relationships)
- **Cross-species edges:** genes in the same eggNOG orthologous group are linked; two weighting schemes tested (uniform vs degree-based)
- **Embedding:** node2vec (random-walk based), chosen over adjacency matrix because it scales to 100K+ genes and naturally brings cross-species orthologs close together
- **Classifier:** simple logistic regression with L2 regularization (deliberately simple — the heavy lifting is in the embedding)
- **Evaluation metric:** log₂(auPRC/prior) — "how many 2× better than random"
- **Holdout scheme:** study-bias split (well-studied genes → train, under-studied → test)

### Key results (the ones that matter for your project)

1. **Multi-species > single-species, always.** Features from all 6 species beat features from human alone for predicting human gene functions — even when the task is purely human. The mouse+human pair alone already gives a substantial boost.

2. **Node embeddings beat adjacency matrices for cross-species transfer.** Embeddings naturally "bring together genes from different species" in the vector space; adjacency matrices keep them separate.

3. **You don't need shared orthologs to transfer knowledge.** This is the headline finding. When trained on human Bardet-Biedl Syndrome (BBS) genes and applied to model species, the framework identified ciliary processes/phenotypes in worms and fish whose relevant gene sets share **zero orthologs** with the human BBS genes. The network context does the work.

4. **Explicit ortholog labeling can hurt.** Adding orthologs of positive examples as extra training positives *decreased* performance for node-embedding classifiers — the cross-species information is already baked into the embedding. This is a practical caution for your mouse→human data augmentation.

5. **The embedding is reusable.** Once computed, the same gene embeddings work for any within- or cross-species classification task — no retraining needed.

### Code & data availability
- Code: [github.com/krishnanlab/GenePlexusZoo](https://github.com/krishnanlab/GenePlexusZoo-Manuscript)
- Pre-computed embeddings: [Zenodo](https://zenodo.org/records/10246207)
- The embeddings are **ready to download and use** — you don't need to re-run the pipeline.

## 2. Paper 2: Nature Methods Review — Computational Strategies for Cross-Species Knowledge Transfer

### What it covers
A comprehensive Perspective reviewing state-of-the-art methods across **four key areas** of cross-species knowledge transfer, plus introduces the concept of **agnology**.

### The four areas (and which ones matter for your project)

**Area 1: Transferring disease and gene annotation knowledge across species** ← **MOST RELEVANT**
- Methods: GenePlexusZoo (network embedding), FKT (functional knowledge transfer), NetQuilt (multi-network integration), phylogenetic profiles, cross-species ontologies (uPheno, PhenomeNET)
- **Your use case:** transfer mouse nAChR variant effect labels to human nAChR genes

**Area 2: Identifying functionally equivalent molecular components** ← **HIGHLY RELEVANT**
- Methods: MUNK/MUNDO (network embedding for homolog mapping), ETNA, protein language models (ESM-2), Phenologs, CoCoCoNet, ModuleBlast, xHeinz
- **Your use case:** determining which mouse nAChR subunit residues are functionally equivalent to which human residues (the position-mapping problem in [[data-scraper-goal]])
- **Key insight from the review:** "Orthology alone does not guarantee functional similarity, and paralogs or even non-homologous genes can exhibit greater functional equivalence." This is directly relevant to mapping mouse→human nAChR positions.

**Area 3: Inferring equivalent perturbed genes or gene sets** ← **MODERATELY RELEVANT**
- Methods: FIT (linear regression on matched cross-species expression), TransComp-R (PCA projection), AutoTransOP (autoencoder style transfer), XGSEA (cross-species GSEA)
- **Your use case:** if you later incorporate transcriptomic data (e.g., from nAChR-related expression studies), these methods could translate mouse expression perturbation signatures to human

**Area 4: Identifying equivalent cell types** ← **LESS RELEVANT (for now)**
- Methods: SAMap, SATURN (uses protein language models — "the only non-transformer method capable of aligning single-cell datasets from multiple species simultaneously"), Precious3GPT, GeneCompass, Universal Cell Embeddings (UCE)
- **Your use case:** only relevant if you later incorporate tissue/cell-type-specific expression context for nAChR variants

### The concept of "Agnology" — and why it matters for nAChRs

The authors introduce **agnology** to describe *functional equivalence of biological entities regardless of their evolutionary origins.* This is a data-driven framework where:

- Functions can remain **evolutionarily unresolved** — you don't need to prove homology to claim functional similarity
- **Orthology ≠ functional equivalence.** Two genes can be orthologs but have diverged in function; two non-homologous genes can converge on the same function
- Modern ML methods (embeddings, networks) naturally capture this — they group by functional similarity in the learned space, not by evolutionary ancestry

**Why this matters for nAChR variant prediction:**
- nAChR subunits (CHRNA1-10, CHRNB1-4, CHRND, CHRNE, CHRNG) have undergone **complex duplication and subfunctionalization** across vertebrates
- Mouse and human nAChR subunits are orthologous, but their **functional equivalence at specific residue positions** is not guaranteed
- A mouse α1 subunit variant at position X may not map cleanly to human α1 position X — the protein context, binding interfaces, and allosteric networks may differ
- The agnology framework gives you a principled way to handle this: **use network context + embedding similarity rather than pure sequence alignment** to decide which mouse variants are informative for which human predictions

## 3. How These Papers Connect to Your Project

### The direct connection: your project already has a cross-species data problem

From [[data-scraper-goal]]: *"The project is a Variant Effect Predictor for human nAChR proteins, but human variant data is insufficient. The plan is to augment the training set with MOUSE nAChR data."*

And the known caveat: *"mouse residue numbering does not map 1:1 to human positions, so folding mouse variants into a human VEP needs a position-alignment step."*

These two papers provide **the computational framework for doing this augmentation rigorously**, going far beyond simple sequence alignment.

### Three concrete ways to use these papers

### 3.1 Use GenePlexusZoo embeddings as features for your VEP model

**What to do:**
1. Download the pre-computed GenePlexusZoo embeddings from Zenodo (human + mouse)
2. For each nAChR gene (CHRNA1, CHRNB2, etc.), extract its 128-dimensional node2vec embedding
3. Add these as features to your existing feature matrix

**Why it should help:**
- The embedding captures the gene's **network context** — what it interacts with, what pathways it's in, what it's functionally similar to — across species
- For nAChR subunits, this encodes: which other subunits they co-assemble with, their synaptic localization context, their signalling pathway membership
- This is information your current features (physicochemical + structural) don't capture
- **Effort: Low.** Embeddings are pre-computed. Just a lookup + concatenation.
- **Novelty: Medium.** Using network features for VEP is done by LoGoFunc, but using *cross-species* network embeddings for ion-channel-specific VEP appears novel.

**Caveat:** The GenePlexusZoo embeddings are at the **gene level**, not the **variant/residue level**. So they would be gene-level context features (same for all variants in CHRNA1, different for CHRNB2, etc.), not residue-specific features. This is still useful — different nAChR subunits have different interaction partners and pathway contexts — but it won't replace residue-level features.

### 3.2 A principled framework for mouse→human variant data augmentation

**The problem, restated:** You scrape mouse nAChR electrophysiology data (LOF/GOF labels). How do you use these to train a human VEP?

**The naive approach (don't do this):** Align mouse and human nAChR sequences with BLAST/Clustal, map residues by alignment position, treat mouse variant at aligned position = human variant at that position, and pool all data together.

**Why the naive approach is risky:**
- nAChR subunits have long intracellular loops (especially the M3-M4 loop) that are poorly conserved even between mouse and human
- Functional equivalence at a position depends on 3D context, not just linear alignment
- A LOF mutation in mouse α1 at position X might be GOF (or neutral) in human α1 at the aligned position, if the local structural environment differs

**The better approach (informed by these papers):**

1. **Use ESM-2 embeddings for residue-level equivalence.** From the Nature Methods review, protein language models (ESM-2) produce embeddings where functionally similar residues across species cluster together, even when sequence identity is low. For each mouse variant position, compute the cosine similarity between its ESM-2 embedding and each human position's embedding → weight the mouse label by this similarity.

2. **Use GenePlexusZoo for gene-level context.** If the mouse gene's embedding is close to the human gene's embedding, the cross-species transfer is more trustworthy. For nAChRs, some subunits (e.g., α1) are highly conserved; others (e.g., α10) are more divergent.

3. **Build a "transfer confidence" score** combining ESM-2 residue similarity + GenePlexusZoo gene similarity + sequence identity at that position. Use this to: (a) filter which mouse variants to include, (b) weight training examples, or (c) define a "cross-species" vs "same-species" evaluation split.

4. **Evaluate cross-species transfer explicitly.** Train on human-only, test on human-only → baseline. Train on human+mouse, test on human → your augmented model. The difference is your cross-species gain. Report this as a separate metric — it's a paper-worthy result in itself.

### 3.3 The agnology concept as a framing device for your paper

The "agnology" concept from the Nature Methods review gives you a **conceptual framework** to justify why cross-species data augmentation for nAChR VEP is a legitimate, scientifically sound approach:

> *"We adopt an agnology-based framework (Yuan et al., 2026) where functional equivalence between mouse and human nAChR variants is established through data-driven embedding similarity rather than strict orthology. This allows us to leverage the richer electrophysiological characterization available for mouse nAChRs while accounting for the evolutionary divergence between species."*

This is the kind of framing that makes reviewers nod rather than ask "but how do you know the mouse variants are equivalent?"

### 3.4 Specific methods from the review that could help with position mapping

The review mentions several methods that go beyond simple sequence alignment for finding functional equivalents:

| Method | What it does | Relevance to your position-mapping problem | Effort |
|---|---|---|---|
| **MUNK / MUNDO** | Network embedding for cross-species homolog mapping | Could identify which mouse residues are functionally equivalent to which human residues using network context, not just sequence | High (requires training) |
| **ESM-2 embeddings** | Protein language model — residue-level representations | Cosine similarity between mouse and human residue embeddings → functional equivalence score. Already in your features.ipynb recommendations. | Low |
| **ETNA** | Network alignment across species | If nAChR interaction networks exist for both species, ETNA could align them at the residue or domain level | High |
| **Phenologs** | Cross-species phenotype equivalence via orthologous gene sets | If mouse nAChR knockout phenotypes are well-characterized, phenolog mapping could validate which mouse subunits best model their human counterparts | Medium |

**Recommendation:** Start with ESM-2 (low effort, already on your radar from `features.ipynb`) and add GenePlexusZoo embeddings. These two together give you residue-level + gene-level cross-species signal. The more complex methods (MUNK, ETNA) are aspirational — cite them as future work.

## 4. Concrete Implementation Plan

### Phase 1: Quick wins (1-2 days)

1. **Download GenePlexusZoo embeddings** from Zenodo for human + mouse
2. **Extract embeddings** for all 16 human nAChR genes (CHRNA1-7, A9, A10, CHRNB1-4, CHRND, CHRNE, CHRNG) and their mouse orthologs
3. **Add as gene-level features** to your feature matrix: for each variant, concatenate the 128-d embedding of its gene
4. **Re-run ablation** (with/without network features) to measure gain

### Phase 2: Cross-species augmentation pipeline (3-5 days)

1. **For each scraped mouse variant:** compute ESM-2 embedding of the mutated residue in its local sequence window (±5 residues)
2. **For each human position** in the aligned subunit: compute ESM-2 embedding of the equivalent window
3. **Compute transfer confidence** = cosine_similarity(mouse_emb, human_emb) × sequence_identity(mouse_window, human_window)
4. **Filter/augment:** include mouse variants with confidence > threshold as weighted training examples for the aligned human position
5. **Evaluate:** human-only baseline vs human+mouse augmented, with leave-one-subunit-out CV

### Phase 3: Paper contribution framing (ongoing)

Frame the cross-species aspect as a secondary contribution alongside the primary nAChR-specific GOF/LOF contribution (from [[vep-research-direction]]). Two contributions:
1. **Primary:** First nAChR-specific GOF/LOF predictor combining pLM embeddings + structural (open/closed) features
2. **Secondary:** Principled cross-species data augmentation using embedding-based functional equivalence (agnology framework), validated by showing that mouse-augmented training improves human variant prediction

## 5. Potential Pitfalls & How to Address Them

### Pitfall 1: Gene-level embeddings don't help residue-level prediction
**Risk:** GenePlexusZoo embeddings are gene-level. If nAChR subunits are too similar to each other in network space (they all interact with each other in the same complex), the embeddings may not add discriminatory power.
**Mitigation:** Check the pairwise cosine distances between nAChR gene embeddings before using them. If CHRNA1 and CHRNB2 embeddings are nearly identical, they won't help distinguish variants. Report this honestly.

### Pitfall 2: Circularity in cross-species evaluation
**Risk:** If you use mouse data in training and some mouse variants map to the same human positions as your human test variants, you've leaked information.
**Mitigation:** The grouped CV by subunit (already in your plan) largely handles this. Additionally: if you ever do a mouse→human transfer, ensure the test human variants are at positions that have NO mouse variant mapped to them, or use a strict temporal split (mouse data collected before date X → train, human variants discovered after date X → test).

### Pitfall 3: ESM-2 itself was trained on sequences that include your test variants
**Risk:** ESM-2 embeddings of your test variants may encode information from the model's training, creating subtle leakage.
**Mitigation:** This is a field-wide problem, not unique to you. Acknowledge it in limitations. Use the "circularity tier" taxonomy from your `vep_research.ipynb` (population-free / population-tuned / clinical-trained) to tag ESM-2 features appropriately.

### Pitfall 4: The mouse data may not add signal
**Risk:** If mouse nAChR electrophysiology is too different from human nAChR function (different subunit composition, different expression context, different post-translational modifications), mouse labels may add noise, not signal.
**Mitigation:** Run a negative control: train with mouse data + random human labels, and show it doesn't beat the human-only baseline. Report the cross-species gain (or lack thereof) honestly — a null result with a careful analysis is still publishable.

## 6. How to Cite These Papers in Your Work

### If you use GenePlexusZoo embeddings as features:
> *"Gene-level network context features were obtained from GenePlexusZoo (Mancuso et al., 2024), a multi-species network embedding framework that projects genes from human and five model organisms into a joint low-dimensional space using node2vec on orthology-connected molecular networks."*

### If you use the agnology framework to justify cross-species augmentation:
> *"Following the agnology framework of Yuan et al. (2026), we assess functional equivalence between mouse and human nAChR variants through embedding similarity rather than strict orthology, acknowledging that orthologous positions may diverge in functional effect while non-homologous positions may converge."*

### If you use ESM-2 for residue-level cross-species mapping:
> *"Residue-level functional equivalence across species was estimated using cosine similarity between ESM-2 (Lin et al., 2023) embeddings of local sequence windows, an approach consistent with the protein language model-based strategies for cross-species component identification reviewed in Yuan et al. (2026)."*

## 7. Summary: The One-Paragraph Take-Away for Your Advisor

These two papers provide the computational framework for rigorously augmenting our nAChR GOF/LOF training data with mouse electrophysiology variants — the exact task our datascraper is doing. GenePlexusZoo (Mancuso et al., 2024) gives us pre-computed cross-species gene embeddings we can use as features immediately; the Nature Methods review (Yuan et al., 2026) gives us (a) a taxonomy of cross-species transfer methods to choose from, (b) the "agnology" concept as a principled justification for why embedding-based functional equivalence beats pure sequence alignment, and (c) specific methods (ESM-2, MUNK, Phenologs) for mapping mouse→human residue positions. The concrete plan: (1) add GenePlexusZoo embeddings as gene-level features now (low effort, pre-computed), (2) build an ESM-2-based transfer confidence score for mouse→human variant mapping, and (3) frame the cross-species augmentation as a secondary paper contribution alongside the nAChR-specific GOF/LOF model. The key risk — that mouse and human nAChR positions are not functionally equivalent — is exactly what the agnology framework is designed to handle.

### Verdict: Both papers are directly actionable. GenePlexusZoo for features; the Nature Methods review for experimental design and paper framing.

## References

1. Mancuso CA, Johnson KA, Liu R, Krishnan A. Joint representation of molecular networks from multiple species improves gene classification. *PLOS Computational Biology* (2024) 20(1):e1011773. https://doi.org/10.1371/journal.pcbi.1011773
2. Yuan H, Mancuso CA, Johnson K, Braasch I, Krishnan A. Computational strategies for cross-species knowledge transfer. *Nature Methods* (2026) 23(2):312–327. https://doi.org/10.1038/s41592-025-02931-9
3. Lin Z, et al. Evolutionary-scale prediction of atomic-level protein structure with a language model. *Science* (2023) 379:1123–1130. (ESM-2)
4. Stein D, et al. Genome-wide prediction of pathogenic gain- and loss-of-function variants from ensemble learning of a diverse feature set. *Genome Medicine* (2023) 15:103. (LoGoFunc — uses network features for GOF/LOF)

---
*Notebook by Claude (2026-07-27). Review of two papers recommended by advisor. Cross-references: [[vep-research-direction]], [[data-scraper-goal]], features.ipynb, week1.ipynb.*